# NUFROST Unified Evaluations (Local)

This notebook runs all major evaluation experiments from the local checkout and writes CSV outputs incrementally.

Experiments included:
- Ablation study
- Sparse observation sweep
- Continuous gap-length sweep
- Repeatability evaluation

The workflow evaluates complete images through loader-owned streaming reads, reuses fixed random samples, and appends results to CSV files under `data/output/`.


In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET_NAME = "hls"  # or "sentinel-2"
IMAGE_DIR = PROJECT_DIR / f"data/{DATASET_NAME}"
OUTPUT_DIR = PROJECT_DIR / "data/output"
CACHE_DIR = PROJECT_DIR / "data/cache/local"
READ_MODE = "streaming"

OUTPUT_PATHS = {
    "ablation": OUTPUT_DIR / f"{DATASET_NAME}_ablation_results.csv",
    "sparse": OUTPUT_DIR / f"{DATASET_NAME}_sparse_sweep_results.csv",
    "gap": OUTPUT_DIR / f"{DATASET_NAME}_gap_sweep_results.csv",
    "repeatability": OUTPUT_DIR / f"{DATASET_NAME}_repeatability_results.csv",
}

IMAGE_NAMES = []
MAX_IMAGES = None
PREBUILD_CACHE = False
PREBUILD_LIMIT = 5
N_JOBS = -1
BASE_SEED = 42

SPARSE_POINT_LEVELS = [1000, 5000, 10000, 20000]
MAX_RANDOM_POINTS = max(SPARSE_POINT_LEVELS)
ABLATION_RANDOM_POINTS = MAX_RANDOM_POINTS

GAP_INDEX_TARGETS = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.22, 0.26, 0.30, 0.35, 0.40, 0.46, 0.52, 0.60, 0.70, 0.80]
MAX_GAP_SAMPLES = None
ABLATION_GAP_INDEX = 0.30
GAP_MAX_MISSING_RATIO = 0.08
GAP_MAX_NATIVE_GAP_DAYS = 60

REPEATABILITY_SEEDS = [11, 23, 37, 53, 71]
REPEATABILITY_IMAGE_LIMIT = 5
REPEATABILITY_RANDOM_POINTS = 10000
REPEATABILITY_GAP_INDEX = 0.30
REPEATABILITY_GAP_SAMPLES = 500

ABLATION_VARIANTS = [
    {"name": "Full NUFROST", "overrides": {}},
    {"name": "w/o preferred frequencies", "overrides": {"frequency_selection": "spectral"}},
    {"name": "w/o parabolic refinement", "overrides": {"refine_peaks": False}},
    {"name": "w/o Huber robust fitting", "overrides": {"huber_iters": 0}},
    {"name": "w/o frequency-weighted ridge", "overrides": {"freq_weight": 0.0}},
    {"name": "w/o linear trend", "overrides": {"include_trend": False}},
]


In [2]:
import os

os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


[Working directory changed to: /Volumes/T7/nuforst]


In [3]:
print("Install notebook/runtime dependencies in your local environment before running this notebook.")
print("Recommended: conda env create -f environment.yml or pip install -r requirements.txt")


Install notebook/runtime dependencies in your local environment before running this notebook.
Recommended: conda env create -f environment.yml or pip install -r requirements.txt


In [4]:
import glob
import importlib
import re
import time
import zlib

import numpy as np
import pandas as pd

import src.data_loader
import src.evaluation
import src.hants
import src.nufrost
import src.zhu2015

from config import build_args

importlib.reload(src.nufrost)
importlib.reload(src.zhu2015)
importlib.reload(src.hants)
importlib.reload(src.evaluation)
importlib.reload(src.data_loader)

def stable_seed(*parts: object) -> int:
    payload = "::".join(str(part) for part in parts).encode("utf-8")
    return BASE_SEED + (zlib.adler32(payload) % 1_000_000)

def loc_id_from_paths(image_paths):
    stem = Path(image_paths[0]).stem
    match = re.search(r"([A-Z0-9]+_lon[0-9.]+_lat[0-9.]+)", stem)
    return match.group(1) if match else stem

def parse_loc_id(loc_id: str):
    match = re.fullmatch(r"([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+)", loc_id)
    if not match:
        return {"Band": None, "Lon": None, "Lat": None}
    return {"Band": match.group(1), "Lon": float(match.group(2)), "Lat": float(match.group(3))}

def append_rows(csv_path: Path, df: pd.DataFrame) -> None:
    if df.empty:
        return
    header = not csv_path.exists()
    df.to_csv(csv_path, mode="a", header=header, index=False)

def log_step(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)

_pixel_stats_cache = {}

def get_pixel_stats(source, t_days, loc_id, min_obs):
    if loc_id not in _pixel_stats_cache:
        log_step(f"Scanning pixel stats for {loc_id} (window-based, no per-pixel I/O)")
        _pixel_stats_cache[loc_id] = src.evaluation.scan_pixel_stats_from_source(source, t_days)
        stats = _pixel_stats_cache[loc_id]
        vc = stats['valid_counts']
        n_valid = int(np.sum(vc >= max(min_obs + 1, 3)))
        n_gap = int(np.sum(vc >= max(min_obs + 3, 15)))
        log_step(f"Pixel stats done: {n_valid} pixels with >={max(min_obs + 1, 3)} obs, {n_gap} with >={max(min_obs + 3, 15)} obs")
    return _pixel_stats_cache[loc_id]

def summarize_gap_candidates(source, t_days, min_obs, loc_id=None):
    stats = get_pixel_stats(source, t_days, loc_id, min_obs) if loc_id else src.evaluation.scan_pixel_stats_from_source(source, t_days)
    return src.evaluation.scan_gap_candidates_from_source(source, t_days, min_obs, max_candidates=50000, seed=BASE_SEED, precomputed_stats=stats)

def select_gap_pixels(source, t_days, min_obs, num_samples, seed, max_missing_ratio, max_native_gap_days, loc_id=None):
    candidates = summarize_gap_candidates(source, t_days, min_obs, loc_id=loc_id)
    filtered = [(r, c) for r, c, missing_ratio, native_gap_days in candidates if missing_ratio <= max_missing_ratio and native_gap_days <= max_native_gap_days]
    if not filtered:
        return np.empty((0, 2), dtype=int), len(candidates), 0
    filtered_arr = np.array(filtered, dtype=int)
    if num_samples is not None and len(filtered_arr) > num_samples:
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(filtered_arr), num_samples, replace=False)
        filtered_arr = filtered_arr[idx]
    return filtered_arr, len(candidates), len(filtered)

def gap_days_from_index_targets(t_days, targets):
    total_span_days = float(np.nanmax(t_days) - np.nanmin(t_days))
    if (not np.isfinite(total_span_days)) or total_span_days <= 0:
        return [], total_span_days
    gap_specs = []
    for idx_val in targets:
        if idx_val <= 0:
            continue
        approx_days = int(round(total_span_days * np.sqrt(idx_val)))
        approx_days = max(1, min(approx_days, int(total_span_days * 0.95)))
        gap_specs.append((approx_days, float(idx_val)))
    dedup = {}
    for gap_days, idx_val in gap_specs:
        dedup[gap_days] = idx_val
    return sorted(dedup.items()), total_span_days

def gap_days_for_index(t_days, index_value):
    gap_specs, total_span_days = gap_days_from_index_targets(t_days, [index_value])
    if not gap_specs:
        return 1, total_span_days
    return gap_specs[0][0], total_span_days

def load_done_keys(csv_path: Path, key_columns):
    if not csv_path.exists():
        return set()
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f"Could not read {csv_path.name}: {exc}")
        return set()
    if any(col not in df.columns for col in key_columns):
        return set()
    return set(tuple(row[col] for col in key_columns) for _, row in df.iterrows())

def _make_batch_checkpoint_writer(csv_path, loc_id, scenario, variant, gap_length=None, gap_index_target=None, loc_meta=None):
    checkpoint_path = csv_path.parent / f"{csv_path.stem}_checkpoint.csv"
    def on_batch_done(batch_results, batch_start, batch_end):
        n_valid = sum(1 for r in batch_results if r is not None)
        if n_valid == 0:
            return
        print(f"  [checkpoint] {scenario}/{variant} pixels {batch_start}-{batch_end}: {n_valid}/{len(batch_results)} valid", flush=True)
        row = {
            "Image": loc_id,
            "Scenario": scenario,
            "Variant": variant,
            "BatchStart": batch_start,
            "BatchEnd": batch_end,
            "ValidPixels": n_valid,
            "Timestamp": pd.Timestamp.now().isoformat(),
        }
        if gap_length is not None:
            row["GapLength"] = gap_length
        if gap_index_target is not None:
            row["GapIndexTarget"] = gap_index_target
        if loc_meta:
            row.update(loc_meta)
        append_rows(checkpoint_path, pd.DataFrame([row]))
    return on_batch_done, checkpoint_path


In [5]:
if IMAGE_NAMES:
    image_paths_list = [[(IMAGE_DIR / name).as_posix()] for name in IMAGE_NAMES]
else:
    files = glob.glob((IMAGE_DIR / "*.tif").as_posix())
    loc_ids = set()
    for file_path in files:
        name = Path(file_path).name
        match = re.search(r"_([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+).*?(?:_part\d+)?(?:-\d{10}-\d{10})?\.tif$", name)
        if match:
            loc_ids.add((match.group(1), float(match.group(2)), float(match.group(3))))

    image_paths_list = []
    for band, lon, lat in sorted(loc_ids):
        chunks = src.data_loader.find_image_chunks(IMAGE_DIR.as_posix(), lon, lat, band, cache_dir=CACHE_DIR.as_posix())
        if chunks:
            image_paths_list.append(chunks)

if MAX_IMAGES is not None:
    image_paths_list = image_paths_list[:MAX_IMAGES]

log_step(f"Found {len(image_paths_list)} distinct spatial/band chunks to evaluate.")
if PREBUILD_CACHE:
    for image_paths in image_paths_list[:PREBUILD_LIMIT]:
        log_step(f"Prebuilding streaming source for {loc_id_from_paths(image_paths)}")
        args = build_args({})
        args.cache_dir = CACHE_DIR.as_posix()
        args.force_refresh = False
        with src.evaluation.open_evaluation_source(image_paths, args) as prepared:
            log_step(f"Streaming source ready for {loc_id_from_paths(image_paths)}: shape=({prepared['meta']['count']}, {prepared['meta']['height']}, {prepared['meta']['width']})")


[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon91.2734_lat29.7904.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon91.9113_lat29.1816.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon92.6035_lat28.9443.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon94.2605_lat29.7733.vrt
[DataLoader] VRT cache miss: /Volumes/T7/nuforst/data/hls/BLUE_lon94.361_lat29.6483.vrt (no matching TIFF files)
[DataLoader] VRT cache miss: /Volumes/T7/nuforst/data/hls/BLUE_lon100.112_lat25.654.vrt (no matching TIFF files)
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon100.7173_lat22.1298.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon101.2663_lat24.3681.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local/vrts/BLUE_lon102.2691_lat27.9628.vrt
[DataLoader] VRT cache hit: /Volumes/T7/nuforst/data/cache/local

In [ ]:
ablation_done = load_done_keys(OUTPUT_PATHS["ablation"], ["Image", "Scenario", "Variant"])
sparse_done = load_done_keys(OUTPUT_PATHS["sparse"], ["Image", "NumPoints"])
gap_done = load_done_keys(OUTPUT_PATHS["gap"], ["Image", "GapLength"])
repeat_done = load_done_keys(OUTPUT_PATHS["repeatability"], ["Image", "Scenario", "RepeatSeed"])
repeatability_targets = {loc_id_from_paths(paths) for paths in image_paths_list[:REPEATABILITY_IMAGE_LIMIT]}

for image_index, image_paths in enumerate(image_paths_list, start=1):
    loc_id = loc_id_from_paths(image_paths)
    loc_meta = parse_loc_id(loc_id)
    chunk_start = time.time()
    log_step(f"=== [{image_index}/{len(image_paths_list)}] {loc_id} ===")

    base_args = build_args({})
    base_args.image = image_paths
    base_args.cache_dir = CACHE_DIR.as_posix()
    base_args.force_refresh = False
    base_args.n_jobs = N_JOBS

    log_step(f"Opening streaming source for {loc_id}")
    with src.evaluation.open_evaluation_source(image_paths, base_args) as prepared:
        source = prepared["source"]
        t_sec = prepared["t_sec"]
        t_days = prepared["t_days"]
        log_step(f"Source ready: shape=({prepared['meta']['count']}, {prepared['meta']['height']}, {prepared['meta']['width']})")

        log_step(f"Sampling random-point pool (max={MAX_RANDOM_POINTS})")
        _stats = get_pixel_stats(source, t_days, loc_id, base_args.min_obs)
        random_points_full = src.evaluation.sample_random_points_from_source(source, t_days, base_args.min_obs, MAX_RANDOM_POINTS, seed=stable_seed(loc_id, "random_pool"), precomputed_stats=_stats)
        log_step(f"Random-point pool ready: {len(random_points_full)} samples")

        log_step("Selecting gap candidates from relatively complete pixels only")
        gap_pixels_full, gap_candidate_total, gap_candidate_filtered = select_gap_pixels(source, t_days, base_args.min_obs, MAX_GAP_SAMPLES, seed=stable_seed(loc_id, "gap_pool"), max_missing_ratio=GAP_MAX_MISSING_RATIO, max_native_gap_days=GAP_MAX_NATIVE_GAP_DAYS, loc_id=loc_id)
        log_step(f"Gap candidates: total={gap_candidate_total}, filtered={gap_candidate_filtered}, sampled={len(gap_pixels_full)} (missing_ratio<={GAP_MAX_MISSING_RATIO:.2f}, native_gap<={GAP_MAX_NATIVE_GAP_DAYS}d)")

        gap_specs_for_chunk, total_span_days = gap_days_from_index_targets(t_days, GAP_INDEX_TARGETS)
        ablation_gap_days, _ = gap_days_for_index(t_days, ABLATION_GAP_INDEX)
        repeatability_gap_days, _ = gap_days_for_index(t_days, REPEATABILITY_GAP_INDEX)
        log_step(f"Gap span planning: total_span={total_span_days:.1f}d, targets={len(GAP_INDEX_TARGETS)}, derived_specs={gap_specs_for_chunk}")
        log_step(f"Representative gap settings: ablation={ablation_gap_days}d (I~{ABLATION_GAP_INDEX:.2f}), repeatability={repeatability_gap_days}d (I~{REPEATABILITY_GAP_INDEX:.2f})")

        if len(random_points_full) == 0 or len(gap_pixels_full) == 0:
            log_step("Skipping chunk because no valid evaluation samples were found.")
            continue

        for variant in ABLATION_VARIANTS:
            variant_name = variant["name"]
            variant_args = build_args(variant["overrides"])
            variant_args.image = image_paths
            variant_args.cache_dir = CACHE_DIR.as_posix()
            variant_args.force_refresh = False
            variant_args.n_jobs = N_JOBS

            random_key = (loc_id, "random", variant_name)
            if random_key not in ablation_done:
                stage_start = time.time()
                log_step(f"Ablation random start: {variant_name} ({ABLATION_RANDOM_POINTS} points)")
                df_random = src.evaluation.evaluate_algorithms_from_source(source, t_sec, t_days, variant_args, sampled_points=random_points_full[:ABLATION_RANDOM_POINTS], n_jobs=N_JOBS)
                df_random = df_random[df_random["Algorithm"] == "NuFrost"].copy()
                df_random["Image"] = loc_id
                df_random["Scenario"] = "random"
                df_random["Variant"] = variant_name
                for key, value in loc_meta.items():
                    df_random[key] = value
                append_rows(OUTPUT_PATHS["ablation"], df_random)
                log_step(f"Ablation random done: {variant_name} in {time.time() - stage_start:.1f}s")
                ablation_done.add(random_key)

            gap_key = (loc_id, "gap", variant_name)
            if gap_key not in ablation_done:
                stage_start = time.time()
                log_step(f"Ablation gap start: {variant_name} ({ablation_gap_days} days, {len(gap_pixels_full)} pixels, I~{ABLATION_GAP_INDEX:.2f})")
                on_batch, ckpt_path = _make_batch_checkpoint_writer(OUTPUT_PATHS["ablation"], loc_id, "gap", variant_name, gap_length=ablation_gap_days, gap_index_target=ABLATION_GAP_INDEX, loc_meta=loc_meta)
                df_gap = src.evaluation.evaluate_timeseries_from_source(source, t_sec, t_days, variant_args, simulate_gap_days=ablation_gap_days, sampled_pixels=gap_pixels_full, n_jobs=N_JOBS, on_batch_done=on_batch)
                df_gap = df_gap[df_gap["Algorithm"] == "NuFrost"].copy()
                df_gap["Image"] = loc_id
                df_gap["Scenario"] = "gap"
                df_gap["Variant"] = variant_name
                df_gap["GapLength"] = ablation_gap_days
                df_gap["GapIndexTarget"] = ABLATION_GAP_INDEX
                for key, value in loc_meta.items():
                    df_gap[key] = value
                append_rows(OUTPUT_PATHS["ablation"], df_gap)
                if ckpt_path.exists():
                    ckpt_path.unlink()
                log_step(f"Ablation gap done: {variant_name} in {time.time() - stage_start:.1f}s")
                ablation_done.add(gap_key)

        baseline_random_key = (loc_id, "random", "Baselines")
        if baseline_random_key not in ablation_done:
            stage_start = time.time()
            log_step(f"Ablation random baseline start ({ABLATION_RANDOM_POINTS} points)")
            df_baseline_random = src.evaluation.evaluate_algorithms_from_source(source, t_sec, t_days, base_args, sampled_points=random_points_full[:ABLATION_RANDOM_POINTS], n_jobs=N_JOBS)
            df_baseline_random = df_baseline_random[df_baseline_random["Algorithm"].isin(["Zhu2015", "HANTS"])].copy()
            df_baseline_random["Image"] = loc_id
            df_baseline_random["Scenario"] = "random"
            df_baseline_random["Variant"] = df_baseline_random["Algorithm"]
            for key, value in loc_meta.items():
                df_baseline_random[key] = value
            append_rows(OUTPUT_PATHS["ablation"], df_baseline_random)
            log_step(f"Ablation random baseline done in {time.time() - stage_start:.1f}s")
            ablation_done.add(baseline_random_key)

        baseline_gap_key = (loc_id, "gap", "Baselines")
        if baseline_gap_key not in ablation_done:
            stage_start = time.time()
            log_step(f"Ablation gap baseline start ({ablation_gap_days} days, {len(gap_pixels_full)} pixels, I~{ABLATION_GAP_INDEX:.2f})")
            on_batch, ckpt_path = _make_batch_checkpoint_writer(OUTPUT_PATHS["ablation"], loc_id, "gap", "Baselines", gap_length=ablation_gap_days, gap_index_target=ABLATION_GAP_INDEX, loc_meta=loc_meta)
            df_baseline_gap = src.evaluation.evaluate_timeseries_from_source(source, t_sec, t_days, base_args, simulate_gap_days=ablation_gap_days, sampled_pixels=gap_pixels_full, n_jobs=N_JOBS, on_batch_done=on_batch)
            df_baseline_gap = df_baseline_gap[df_baseline_gap["Algorithm"].isin(["Zhu2015", "HANTS"])].copy()
            df_baseline_gap["Image"] = loc_id
            df_baseline_gap["Scenario"] = "gap"
            df_baseline_gap["Variant"] = df_baseline_gap["Algorithm"]
            df_baseline_gap["GapLength"] = ablation_gap_days
            df_baseline_gap["GapIndexTarget"] = ABLATION_GAP_INDEX
            for key, value in loc_meta.items():
                df_baseline_gap[key] = value
            append_rows(OUTPUT_PATHS["ablation"], df_baseline_gap)
            if ckpt_path.exists():
                ckpt_path.unlink()
            log_step(f"Ablation gap baseline done in {time.time() - stage_start:.1f}s")
            ablation_done.add(baseline_gap_key)

        for num_points in SPARSE_POINT_LEVELS:
            sparse_key = (loc_id, num_points)
            if sparse_key in sparse_done:
                continue
            stage_start = time.time()
            log_step(f"Sparse sweep start: {num_points} points")
            df_sparse = src.evaluation.evaluate_algorithms_from_source(source, t_sec, t_days, base_args, sampled_points=random_points_full[:num_points], n_jobs=N_JOBS)
            df_sparse["Image"] = loc_id
            df_sparse["NumPoints"] = num_points
            for key, value in loc_meta.items():
                df_sparse[key] = value
            append_rows(OUTPUT_PATHS["sparse"], df_sparse)
            log_step(f"Sparse sweep done: {num_points} points in {time.time() - stage_start:.1f}s")
            sparse_done.add(sparse_key)

        for gap_days, gap_index_target in gap_specs_for_chunk:
            gap_key = (loc_id, gap_days)
            if gap_key in gap_done:
                continue
            stage_start = time.time()
            log_step(f"Gap sweep start: {gap_days} days on {len(gap_pixels_full)} pixels (target I~{gap_index_target:.2f})")
            on_batch, ckpt_path = _make_batch_checkpoint_writer(OUTPUT_PATHS["gap"], loc_id, "gap", f"GapSweep_{gap_days}d", gap_length=gap_days, gap_index_target=gap_index_target, loc_meta=loc_meta)
            df_gap_sweep = src.evaluation.evaluate_timeseries_from_source(source, t_sec, t_days, base_args, simulate_gap_days=gap_days, sampled_pixels=gap_pixels_full, n_jobs=N_JOBS, on_batch_done=on_batch)
            df_gap_sweep["Image"] = loc_id
            df_gap_sweep["GapLength"] = gap_days
            df_gap_sweep["GapIndexTarget"] = gap_index_target
            for key, value in loc_meta.items():
                df_gap_sweep[key] = value
            append_rows(OUTPUT_PATHS["gap"], df_gap_sweep)
            if ckpt_path.exists():
                ckpt_path.unlink()
            log_step(f"Gap sweep done: {gap_days} days in {time.time() - stage_start:.1f}s")
            gap_done.add(gap_key)

        if loc_id in repeatability_targets:
            for repeat_seed in REPEATABILITY_SEEDS:
                repeat_random_key = (loc_id, "random", repeat_seed)
                if repeat_random_key not in repeat_done:
                    stage_start = time.time()
                    log_step(f"Repeatability random start: seed={repeat_seed}")
                    sampled_points = src.evaluation.sample_random_points_from_source(source, t_days, base_args.min_obs, REPEATABILITY_RANDOM_POINTS, seed=stable_seed(loc_id, "repeat_random", repeat_seed))
                    df_repeat_random = src.evaluation.evaluate_algorithms_from_source(source, t_sec, t_days, base_args, sampled_points=sampled_points, n_jobs=N_JOBS)
                    df_repeat_random["Image"] = loc_id
                    df_repeat_random["Scenario"] = "random"
                    df_repeat_random["RepeatSeed"] = repeat_seed
                    for key, value in loc_meta.items():
                        df_repeat_random[key] = value
                    append_rows(OUTPUT_PATHS["repeatability"], df_repeat_random)
                    log_step(f"Repeatability random done: seed={repeat_seed} in {time.time() - stage_start:.1f}s")
                    repeat_done.add(repeat_random_key)

                repeat_gap_key = (loc_id, "gap", repeat_seed)
                if repeat_gap_key not in repeat_done:
                    stage_start = time.time()
                    log_step(f"Repeatability gap start: seed={repeat_seed}")
                    sampled_pixels, _, filtered_count = select_gap_pixels(source, t_days, base_args.min_obs, REPEATABILITY_GAP_SAMPLES, seed=stable_seed(loc_id, "repeat_gap", repeat_seed), max_missing_ratio=GAP_MAX_MISSING_RATIO, max_native_gap_days=GAP_MAX_NATIVE_GAP_DAYS, loc_id=loc_id)
                    log_step(f"Repeatability gap sampled {len(sampled_pixels)} pixels from {filtered_count} filtered candidates")
                    on_batch, ckpt_path = _make_batch_checkpoint_writer(OUTPUT_PATHS["repeatability"], loc_id, "gap", f"Repeat_seed{repeat_seed}", gap_length=repeatability_gap_days, gap_index_target=REPEATABILITY_GAP_INDEX, loc_meta=loc_meta)
                    df_repeat_gap = src.evaluation.evaluate_timeseries_from_source(source, t_sec, t_days, base_args, simulate_gap_days=repeatability_gap_days, sampled_pixels=sampled_pixels, n_jobs=N_JOBS, on_batch_done=on_batch)
                    df_repeat_gap["Image"] = loc_id
                    df_repeat_gap["Scenario"] = "gap"
                    df_repeat_gap["RepeatSeed"] = repeat_seed
                    df_repeat_gap["GapLength"] = repeatability_gap_days
                    df_repeat_gap["GapIndexTarget"] = REPEATABILITY_GAP_INDEX
                    for key, value in loc_meta.items():
                        df_repeat_gap[key] = value
                    append_rows(OUTPUT_PATHS["repeatability"], df_repeat_gap)
                    if ckpt_path.exists():
                        ckpt_path.unlink()
                    log_step(f"Repeatability gap done: seed={repeat_seed} in {time.time() - stage_start:.1f}s")
                    repeat_done.add(repeat_gap_key)

        log_step(f"Chunk complete: {loc_id} in {time.time() - chunk_start:.1f}s")

print("\nAll evaluations finished.")
for name, path in OUTPUT_PATHS.items():
    print(f"- {name}: {path}")


[20:02:26] === [1/67] BLUE_lon91.2734_lat29.7904 ===
[20:02:26] Opening streaming source for BLUE_lon91.2734_lat29.7904
[20:02:29] Source ready: shape=(14202, 1002, 1147)
[20:02:29] Sampling random-point pool (max=20000)
[20:02:29] Scanning pixel stats for BLUE_lon91.2734_lat29.7904 (window-based, no per-pixel I/O)
  scan_pixel_stats: 2/20 windows
  scan_pixel_stats: 4/20 windows
  scan_pixel_stats: 6/20 windows
  scan_pixel_stats: 8/20 windows
  scan_pixel_stats: 10/20 windows
  scan_pixel_stats: 12/20 windows
  scan_pixel_stats: 14/20 windows
  scan_pixel_stats: 16/20 windows
  scan_pixel_stats: 18/20 windows
[20:10:47] Pixel stats done: 1149294 pixels with >=13 obs, 1149294 with >=15 obs
[20:12:42] Random-point pool ready: 20000 samples
[20:12:42] Selecting gap candidates from relatively complete pixels only
[20:12:42] Gap candidates: total=50000, filtered=0, sampled=0 (missing_ratio<=0.08, native_gap<=20d)
[20:12:42] Gap span planning: total_span=4090.0d, targets=18, derived_specs=

In [ ]:
for name, csv_path in OUTPUT_PATHS.items():
    if not csv_path.exists():
        print(f"{name}: no results yet")
        continue
    df = pd.read_csv(csv_path)
    print(f"\n{name}: {len(df)} rows")
    display(df.head())
